In [ ]:
#Import all the packages as needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import seaborn as sns


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
#Load the raw file with Office names, population, service volume, distance, service operation hours and SEI data. 
df = pd.read_excel("SBC Peer Office Check.xlsx",header=1)
df.head()


In [ ]:
#Function for standardazation and data visualization 
def plot_feature_space_distribution(df: pd.DataFrame):
    feature_cols = [
        # 'ServiceEst_pop2030_excludingICBC_SDPR ',
       ' ServicePop2030  (SBC calcuation)',
       # 'Service per capita_excludingICBC&SDPR',
       # 'Weighted_avg_travel_distance_db_AfterPilot',
       # 'Office operation hours per day(raw office operation )',
       # 'Average service per Office operation hours/day',
       'Start SEI on CSD level, Need to figure out mapping on DA level'
    ]

    # --- Step 1: standardize ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[feature_cols])

    df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

    # --- Step 2: histogram for each feature ---
    df_scaled.hist(bins=15, figsize=(10, 6))
    plt.suptitle("Standardized Feature Distributions")
    plt.tight_layout()
    plt.show()

    # --- Step 3: pairplot (relationships) ---
    sns.pairplot(df_scaled)
    plt.suptitle("Feature Space Pairwise Distribution", y=1.02)
    plt.show()

    # --- Step 4: correlation heatmap ---
    plt.figure(figsize=(6, 5))
    sns.heatmap(df_scaled.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Feature Correlation (Standardized)")
    plt.show()

    return df_scaled

df_scaled = plot_feature_space_distribution(df)
df_scaled

In [ ]:
df['OfficeName_includingMission'].unique()

In [ ]:
df.columns

In [ ]:
#Similirarity score using all features
from sklearn.preprocessing import StandardScaler


def find_similar_offices(df, office_name, top_n=10):
    model_df = df.copy()

    # Change: clean column names
    model_df.columns = (
        model_df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    office_col = "OfficeName_includingMission"

    feature_cols = [
        # "ServiceEst_pop2030_excludingICBC_SDPR",
        "ServicePop2030 (SBC calcuation)",
        # "Service per capita_excludingICBC&SDPR",
         "Weighted_avg_travel_distance_db_AfterPilot",
        # "Office operation hours per day(raw office operation )",
        # "Average service per Office operation hours/day",
        "Start SEI on CSD level, Need to figure out mapping on DA level",
    ]

    missing_cols = [c for c in [office_col] + feature_cols if c not in model_df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}\nAvailable: {model_df.columns.tolist()}")

    model_df[office_col] = model_df[office_col].astype(str).str.strip()

    for col in feature_cols:
        model_df[col] = (
            model_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"nan": np.nan, "": np.nan})
        )
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

    # Change: do not drop offices with partial missing values
    # Change: impute missing numeric features using column median
    for col in feature_cols:
        model_df[col] = model_df[col].fillna(model_df[col].median())

    input_clean = office_name.strip().lower()

    exact_matches = model_df.index[
        model_df[office_col].str.lower() == input_clean
    ].tolist()

    partial_matches = model_df.index[
        model_df[office_col].str.lower().str.contains(input_clean, na=False)
    ].tolist()

    matches = exact_matches if exact_matches else partial_matches

    if not matches:
        possible_matches = model_df[
            model_df[office_col].str.lower().str.contains(input_clean[:4], na=False)
        ][office_col].tolist()

        raise ValueError(
            f"Office '{office_name}' not found.\nPossible matches: {possible_matches[:10]}"
        )

    target_idx = matches[0]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(model_df[feature_cols])

    target_vector = X_scaled[target_idx]

    results = []

    for i in range(len(model_df)):
        if i == target_idx:
            continue

        diff = target_vector - X_scaled[i]

        distance = np.linalg.norm(diff)
        similarity_score = 1 / (1 + distance)

        squared_diff = diff ** 2
        total_sq = squared_diff.sum()
        contribution_pct = squared_diff / total_sq if total_sq != 0 else np.zeros_like(diff)

        contribution_df = pd.DataFrame({
            "feature": feature_cols,
            "contribution_pct": contribution_pct,
        }).sort_values("contribution_pct", ascending=False)

        top_impact_features = contribution_df.head(3).to_dict(orient="records")

        results.append({
            "input_office": model_df.loc[target_idx, office_col],
            "similar_office": model_df.loc[i, office_col],
            "distance": round(float(distance), 6),
            "similarity_score": round(float(similarity_score), 6),

            "impact_feature_1": top_impact_features[0]["feature"],
            "impact_pct_1": round(float(top_impact_features[0]["contribution_pct"]), 4),

            "impact_feature_2": top_impact_features[1]["feature"],
            "impact_pct_2": round(float(top_impact_features[1]["contribution_pct"]), 4),

            # "impact_feature_3": top_impact_features[2]["feature"],
            # "impact_pct_3": round(float(top_impact_features[2]["contribution_pct"]), 4),
        })

    result_df = (
        pd.DataFrame(results)
        .sort_values("distance", ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )

    return result_df

In [ ]:
# Return the similarity score for mission office using all three features 
result_df = find_similar_offices(
    df=df,
    office_name="mission",
    top_n=10
)

#result_df

In [ ]:
# Export the result to excel file 
result_df.to_excel("Mission_Comparabale office(Pop+SEI).xlsx", index=False)

# Similiartiy score for each feature and all

In [ ]:
# Function for finding similar offices
# Inputs are dataframe, the office for similarity check. 
def find_top_similar_by_feature_and_all(df, office_name, top_n=10):
    df = df.copy()

    # Clean column names
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    office_col = "OfficeName_includingMission"

    feature_cols = [
        "ServiceEst_pop2030_excludingICBC_SDPR",
        "ServicePop2030 (SBC calcuation)",
        "Service per capita_excludingICBC&SDPR",
        "Weighted_avg_travel_distance_db_AfterPilot",
        "Office operation hours per day(raw office operation )",
        "Average service per Office operation hours/day",
        "Start SEI on CSD level, Need to figure out mapping on DA level",
    ]

    required_cols = [office_col] + feature_cols
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Missing columns: {missing_cols}\nAvailable columns: {df.columns.tolist()}"
        )

    df[office_col] = df[office_col].astype(str).str.strip()

    for col in feature_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Impute missing values to avoid dropping Mission
        df[col] = df[col].fillna(df[col].median())

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[feature_cols])
    X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)

    input_clean = office_name.strip().lower()

    exact_matches = df.index[
        df[office_col].str.lower() == input_clean
    ].tolist()

    partial_matches = df.index[
        df[office_col].str.lower().str.contains(input_clean, na=False)
    ].tolist()

    matches = exact_matches if exact_matches else partial_matches

    if not matches:
        possible = df[
            df[office_col].str.lower().str.contains(input_clean[:4], na=False)
        ][office_col].tolist()

        raise ValueError(
            f"Office '{office_name}' not found. Possible matches: {possible[:10]}"
        )

    target_idx = matches[0]
    target_office = df.loc[target_idx, office_col]

    output = []

    # Similarity by each feature
    for feature in feature_cols:
        target_value = X_scaled_df.loc[target_idx, feature]

        rows = []

        for i in df.index:
            if i == target_idx:
                continue

            distance = abs(target_value - X_scaled_df.loc[i, feature])
            similarity_score = 1 / (1 + distance)

            rows.append({
                "similarity_basis": feature,
                "rank": None,
                "input_office": target_office,
                "similar_office": df.loc[i, office_col],
                "distance": round(float(distance), 6),
                "similarity_score": round(float(similarity_score), 6),
                "input_raw_value": df.loc[target_idx, feature],
                "similar_office_raw_value": df.loc[i, feature],
            })

        temp = (
            pd.DataFrame(rows)
            .sort_values("distance", ascending=True)
            .head(top_n)
            .reset_index(drop=True)
        )

        temp["rank"] = temp.index + 1
        output.append(temp)

    # Similarity by all features combined
    target_pos = df.index.get_loc(target_idx)
    target_vector = X_scaled[target_pos]

    rows = []

    for pos, i in enumerate(df.index):
        if i == target_idx:
            continue

        diff = target_vector - X_scaled[pos]
        distance = np.linalg.norm(diff)
        similarity_score = 1 / (1 + distance)

        rows.append({
            "similarity_basis": "ALL_FEATURES_COMBINED",
            "rank": None,
            "input_office": target_office,
            "similar_office": df.loc[i, office_col],
            "distance": round(float(distance), 6),
            "similarity_score": round(float(similarity_score), 6),
            "input_raw_value": None,
            "similar_office_raw_value": None,
        })

    temp = (
        pd.DataFrame(rows)
        .sort_values("distance", ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )

    temp["rank"] = temp.index + 1
    output.append(temp)

    return pd.concat(output, ignore_index=True)

In [ ]:
# Find the top similar offices by each feature/ all features 
result_df = find_top_similar_by_feature_and_all(
    df=df,
    office_name="mission",
    top_n=10
)

result_df

In [ ]:
# Export the result to excel file. 
result_df.to_excel("Mission_Similarity_Check.xlsx", index=False)